# 03 · Diagnóstico de calidad — Capa Silver (Ingresantes y Matriculados)

**Objetivo:** validar de forma exhaustiva la calidad de los datos de la capa Silver de TUNI.pe — `data/Silver/ingresantes_clean.parquet` y `data/Silver/matriculados_clean.parquet` — y esbozar la **relación conceptual** entre ambos datasets como preparación para el modelo estrella (sin construir tablas Gold).

**Estrategia de memoria (16 GB RAM):**
- **Carga secuencial**: primero Ingresantes (validaciones 2.1–2.8), se libera con `del ing; gc.collect()`, y después Matriculados. **Nunca están ambos datasets en RAM.**
- **Backend Arrow**: `pd.read_parquet(..., engine="pyarrow", dtype_backend="pyarrow")` mantiene los datos compactos.
- **Sin `info(memory_usage="deep")`**: se reportan `shape`, tamaño en disco y una estimación ligera de memoria.
- **Sección 3 con agregados**: se guardan agregaciones pequeñas por dataset y la relación se calcula solo sobre ellas.

**Reglas del notebook:**
- **Solo lectura**: no se modifica ni se escribe nada en `data/Silver/` ni en `data/Gold/`.
- **Basado en contratos**: las claves candidatas se leen de `data/schemas/*.json`.
- **Criterios de aceptación**: 0 nulos en columnas críticas, 0 duplicados bajo la clave efectiva, textos en MAYÚSCULAS sin tildes, periodos 2020–2025 completos y presencia de UCSP en todos los periodos.
- **Autocontenido**: ejecutable de principio a fin.

> **Hallazgo previsto (Ingresantes):** el contrato `ingresantes_schema.json` reporta `duplicados_con_clave = 0`, pero la Silver contiene **293 duplicados residuales** bajo la clave candidata. El notebook los reporta fielmente y los marca como acción correctiva.


In [1]:
from pathlib import Path

# Detección automática de la raíz del proyecto (misma lógica que los notebooks 01 y 02)
current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'data'. Ejecuta este notebook desde la raíz "
        "del proyecto o desde notebooks/."
    )

SILVER = PROJECT_ROOT / "data" / "Silver"
SCHEMAS = PROJECT_ROOT / "data" / "schemas"

ING_PATH = SILVER / "ingresantes_clean.parquet"
MAT_PATH = SILVER / "matriculados_clean.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INGRESANTES existe:", ING_PATH.exists())
print("MATRICULADOS existe:", MAT_PATH.exists())
print("SCHEMAS existe:", SCHEMAS.exists())


PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
INGRESANTES existe: True
MATRICULADOS existe: True
SCHEMAS existe: True


> ⚠️ **Memoria:** `matriculados_clean.parquet` (~736 MB) ocupa ≈ **8 GB de RSS** al cargarse con backend Arrow. Con la carga secuencial, el pico se mantiene ≈ **9 GB**, seguro para una máquina de 16 GB.


In [2]:
import gc
import json
import os
import re
import resource

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

print("pandas", pd.__version__)


def rss_actual_gb():
    """RSS actual del proceso en GB (Linux, /proc/self/statm)."""
    try:
        with open("/proc/self/statm", encoding="utf-8") as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf("SC_PAGE_SIZE") / (1024**3)
    except (OSError, ValueError, IndexError):
        return float("nan")


# ---------------------------------------------------------------------------
# Configuración de diagnóstico y estado compartido
# ---------------------------------------------------------------------------
REGION_SUR = {"AREQUIPA", "CUSCO", "TACNA", "PUNO", "MOQUEGUA", "APURIMAC"}
CODIGO_INEI_UCSP = "260000062"  # Universidad Católica San Pablo
CRITICAS_ING = ["CODIGO_INEI", "GUID_PERSONA", "CODIGO_SIU_PROGRAMA", "PROCESO_ESTANDARIZADO"]
CRITICAS_MAT = ["CODIGO_INEI", "GUID_PERSONA", "CODIGO_SIU_PROGRAMA", "PERIODO_ESTANDARIZADO"]

RESULTADOS = {}  # indicadores finales por dataset (para el resumen ejecutivo)
aggs = {}        # agregaciones pequeñas (para la sección 3)


def cargar_dataset(path, nombre):
    """Carga un Parquet con backend Arrow compacto y muestra métricas ligeras."""
    df = pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
    print("=" * 78)
    print(f"{nombre} cargado")
    print("=" * 78)
    print("Shape:", f"{df.shape[0]:,} filas × {df.shape[1]} columnas")
    print("Archivo en disco:", f"{path.stat().st_size / 1e9:.2f} GB")
    print("Memoria estimada (deep):", f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    print("RSS actual del proceso:", f"{rss_actual_gb():.2f} GB")
    return df


def tabla_nulos(df):
    """Tabla de conteo y % de nulos por columna (de mayor a menor)."""
    nulos = df.isna().sum().sort_values(ascending=False)
    res = pd.DataFrame({
        "columna": nulos.index,
        "nulos": nulos.values,
        "%_nulos": (nulos.values / len(df) * 100).round(2),
    })
    return res[res["nulos"] > 0].reset_index(drop=True)


def resaltar_mayor_5pct(df_res):
    """Resalta en rojo las filas (columnas) con >5% de nulos."""
    def _color(fila):
        if fila["%_nulos"] > 5:
            return ["background-color: #ffd9d9; font-weight: bold"] * len(fila)
        return [""] * len(fila)
    return df_res.style.apply(_color, axis=1)


def verificar_criticas(df, claves):
    """Verifica que las columnas críticas tengan 0 nulos."""
    nulos = df[claves].isna().sum()
    ok = True
    for c in claves:
        cumple = nulos[c] == 0
        ok = ok and cumple
        print(f"  {c}: {nulos[c]:,} nulos  {'✅ OK' if cumple else '❌ INCUMPLE'}")
    return ok


def verificar_duplicados(df, clave, clave_contrato=None):
    """Duplicados bajo una clave. Si se pasa clave_contrato, muestra ambas métricas."""
    total = len(df)
    if clave_contrato is None:
        dup = total - len(df.drop_duplicates(subset=clave))
        print(f"Duplicados bajo la clave ({len(clave)} cols): {dup:,}")
        return dup
    dup_contrato = total - len(df.drop_duplicates(subset=clave_contrato))
    dup_prod = total - len(df.drop_duplicates(subset=clave))
    tabla = pd.DataFrame({
        "clave": [
            "Contrato (6 cols, SIN periodo)",
            "Producción (7 cols, CON periodo)",
        ],
        "n_columnas": [len(clave_contrato), len(clave)],
        "duplicados": [dup_contrato, dup_prod],
        "%_sobre_total": [round(dup_contrato / total * 100, 2), round(dup_prod / total * 100, 2)],
    })
    display(tabla)
    return dup_prod


def verificar_normalizacion(df, cols_muestra):
    """Comprueba MAYÚSCULAS, sin tildes, sin espacios en bordes ni dobles."""
    cols_str = list(df.select_dtypes(include=["object", "string"]).columns)
    filas = []
    for c in cols_str:
        s = df[c]
        filas.append({
            "columna": c,
            "tildes": int(s.str.contains(r"[À-ÿ]", regex=True, na=False).sum()),
            "minúsculas": int(s.str.contains(r"[a-z]", regex=True, na=False).sum()),
            "espacios_borde": int((s != s.str.strip()).sum()),
            "espacios_dobles": int(s.str.contains(r"\s{2,}", regex=True, na=False).sum()),
            "vacíos": int(s.str.strip().eq("").sum()),
        })
    res = pd.DataFrame(filas)
    display(res)
    graves = int(res[["tildes", "minúsculas", "espacios_borde"]].sum().sum())
    menores = int(res["espacios_dobles"].sum())
    if graves == 0 and menores == 0:
        print("→ ✅ Normalización completa (MAYÚSCULAS, sin tildes, sin espacios).")
    elif graves == 0:
        print(f"→ ⚠️ Normalización casi completa: {menores} celdas con espacios dobles (menor).")
    else:
        print(f"→ ❌ Hay {graves} celdas con tildes/minúsculas/espacios en bordes.")
    print("\nMuestra aleatoria (5 valores, random_state=42):")
    for c in cols_muestra:
        print(f"\n[{c}]")
        print(df[c].sample(5, random_state=42).tolist())
    return graves, menores


def validar_region_sur(df, col_dept):
    """Valida Region_Sur contra el departamento real (solo departamentos del SUR → True)."""
    dept = df[col_dept]
    sur_dept = dept.isin(REGION_SUR)
    print("Distribución de Region_Sur (conteo y %):")
    dist = df["Region_Sur"].value_counts().rename("conteo").to_frame()
    dist["%"] = (dist["conteo"] / len(df) * 100).round(2)
    display(dist)
    contrad_sur = int((sur_dept & ~df["Region_Sur"]).sum())
    contrad_no_sur = int((~sur_dept & df["Region_Sur"]).sum())
    print("Contradicciones (depto del SUR pero Region_Sur=False):", contrad_sur)
    print("Contradicciones (depto NO sur pero Region_Sur=True):", contrad_no_sur)
    print("→", "✅ Lógica Region_Sur consistente con los departamentos del SUR"
             if (contrad_sur + contrad_no_sur) == 0 else "❌ Inconsistencias detectadas")
    print("\nConteo por departamento vs Region_Sur:")
    tab = df[[col_dept, "Region_Sur"]].value_counts().unstack(fill_value=0).rename(
        columns={False: "Region_Sur=False", True: "Region_Sur=True"})
    display(tab)
    return contrad_sur + contrad_no_sur


def resumen_periodos(df, col, patron=None, esperados=None):
    """Valores únicos, nulos, formato y cobertura del rango esperado."""
    display(df[col].value_counts().sort_index().rename_axis(col).rename("conteo").to_frame())
    print("Valores únicos:", f"{df[col].nunique():,}", "| Nulos:", f"{int(df[col].isna().sum()):,}")
    if patron is not None:
        mal = int((~df[col].str.match(patron, na=False)).sum())
        print("Valores malformados (no cumplen", patron, "):", mal)
    if esperados is not None:
        obtenidos = set(df[col].dropna().unique())
        faltan = sorted(set(esperados) - obtenidos)
        sobra = sorted(obtenidos - set(esperados))
        if not faltan and not sobra:
            print(f"✅ Cobertura completa: los {len(esperados)} periodos esperados están presentes")
        else:
            print("❌ Faltan:", faltan, "| Sobran:", sobra)


def verificar_ucsp(df, col_periodo, periodos_esperados):
    """Presencia de UCSP y cobertura de periodos."""
    ucsp = df[df["CODIGO_INEI"] == CODIGO_INEI_UCSP]
    nombre = ucsp["NOMBRE_ENTIDAD"].iloc[0] if len(ucsp) else "NO ENCONTRADO"
    print("UCSP →", nombre, "| CODIGO_INEI:", CODIGO_INEI_UCSP)
    print("Filas UCSP:", f"{len(ucsp):,}")
    presentes = set(ucsp[col_periodo].unique())
    faltan = [p for p in periodos_esperados if p not in presentes]
    print(f"Periodos esperados: {len(periodos_esperados)} | Presentes en UCSP: {len(presentes)}")
    print("→", "✅ UCSP tiene datos en TODOS los periodos esperados" if not faltan
             else f"❌ Faltan periodos: {faltan}")
    display(ucsp[col_periodo].value_counts().sort_index().rename_axis(col_periodo).rename("conteo").to_frame())
    return len(ucsp)


def frecuencias(df, cols):
    """Tablas de frecuencia (conteo y %) para variables categóricas clave."""
    for c in cols:
        print(f"\n[{c}]")
        tab = df[c].value_counts(dropna=False).rename("conteo").to_frame()
        tab["%"] = (tab["conteo"] / len(df) * 100).round(2)
        display(tab)


pandas 3.0.5


In [3]:
# Contratos de esquema: claves candidatas para la validación de duplicados
with (SCHEMAS / "ingresantes_schema.json").open("r", encoding="utf-8") as fh:
    schema_ing = json.load(fh)
with (SCHEMAS / "matriculados_schema.json").open("r", encoding="utf-8") as fh:
    schema_mat = json.load(fh)

CLAVE_ING = schema_ing["granularidad"]["clave_candidata"]
CLAVE_MAT_CONTRATO = schema_mat["granularidad"]["clave_candidata"]
COL_PERIODO_MAT = schema_mat["columna_periodo"]
CLAVE_MAT_PROD = [COL_PERIODO_MAT] + CLAVE_MAT_CONTRATO

print("Ingresantes · clave del contrato (3 cols):", CLAVE_ING)
print("Matriculados · clave del contrato (6 cols, SIN periodo):", CLAVE_MAT_CONTRATO)
print("Matriculados · clave de producción (7 cols, CON periodo):", CLAVE_MAT_PROD)


Ingresantes · clave del contrato (3 cols): ['CODIGO_INEI', 'GUID_PERSONA', 'CODIGO_SIU_PROGRAMA']
Matriculados · clave del contrato (6 cols, SIN periodo): ['CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']
Matriculados · clave de producción (7 cols, CON periodo): ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']


## 2. Diagnóstico por dataset

Las validaciones 2.1–2.8 se ejecutan **secuencialmente**: Ingresantes primero (Fase 1) y Matriculados después (Fase 2). Al terminar cada fase se libera la memoria antes de cargar el siguiente dataset.

### 2. Ingresantes (`ingresantes_clean.parquet`)

Dataset **anual** (granularidad: persona × programa × año). Clave candidata del contrato: `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA`.


In [4]:
# Carga secuencial · FASE 1: INGRESANTES
ing = cargar_dataset(ING_PATH, "INGRESANTES")


INGRESANTES cargado
Shape: 3,120,287 filas × 33 columnas
Archivo en disco: 0.14 GB
Memoria estimada (deep): 1.43 GB
RSS actual del proceso: 2.15 GB


#### 2.1 Ingresantes · Resumen general
Columnas, tipos de dato, filas totales y rango de periodos (años).


In [5]:
print("Filas totales:", f"{len(ing):,}", "| Columnas:", ing.shape[1])
print("\nTipos de dato por columna:")
print(ing.dtypes.to_string())
print("\nRango de periodos (PROCESO_ESTANDARIZADO):", sorted(ing["PROCESO_ESTANDARIZADO"].dropna().unique()))


Filas totales: 3,120,287 | Columnas: 33

Tipos de dato por columna:
CODIGO_INEI                         string[pyarrow]
NOMBRE_ENTIDAD                      string[pyarrow]
TIPO_ENTIDAD                        string[pyarrow]
TIPO_GESTION                        string[pyarrow]
LICENCIADO                          string[pyarrow]
TIPO_CONSTITUCION                   string[pyarrow]
NIVEL_ACADEMICO                     string[pyarrow]
PROCESO_ESTANDARIZADO                int64[pyarrow]
GUID_PERSONA                        string[pyarrow]
SEXO                                string[pyarrow]
NACIONALIDAD                        string[pyarrow]
DEPARTAMENTO_NACIMIENTO             string[pyarrow]
ANIO_NACIMIENTO                     double[pyarrow]
EDAD                                string[pyarrow]
CODIGO_SIU_PROGRAMA                  int64[pyarrow]
CODIGO_GRUPO_1                      double[pyarrow]
NOMBRE_GRUPO_1                      string[pyarrow]
CODIGO_GRUPO_3                      double[pyarr

#### 2.2 Ingresantes · Análisis de nulos
Conteo y % de nulos por columna (de mayor a menor). Las columnas con **nulos > 5%** se resaltan en rojo. Columnas críticas (`CODIGO_INEI`, `GUID_PERSONA`, `CODIGO_SIU_PROGRAMA`, `PROCESO_ESTANDARIZADO`) deben tener **0 nulos**.


In [6]:
res = tabla_nulos(ing)
if len(res) == 0:
    print("✅ Ninguna columna con nulos.")
else:
    print("Columnas con nulos (de mayor a menor %); >5% resaltado en rojo:")
    display(resaltar_mayor_5pct(res))

print("\nVerificación de columnas críticas (deben tener 0 nulos):")
verificar_criticas(ing, CRITICAS_ING)


Columnas con nulos (de mayor a menor %); >5% resaltado en rojo:


,columna,nulos,%_nulos
0,DES_DISCAPACIDAD_DE_DISPOSICION,3106905,99.570000
1,DES_DISCAPACIDAD_DE_SITUACION,3106905,99.570000
2,DES_DISCAPACIDAD_DE_DESTREZA,3106905,99.570000
3,DES_DISCAPACIDAD_DE_LOCOMOCION,3106905,99.570000
4,DES_DISCAPACIDAD_DEL_CUIDADO,3106904,99.570000
5,DES_DISCAPACIDAD_DE_CONDUCTA,3106904,99.570000
6,DES_DISCAPACIDAD_DE_COMUNICACION,3106904,99.570000
7,CODIGO_GRUPO_3,163596,5.240000
8,CODIGO_GRUPO_1,163596,5.240000
9,DEPARTAMENTO_NACIMIENTO,39199,1.260000



Verificación de columnas críticas (deben tener 0 nulos):
  CODIGO_INEI: 0 nulos  ✅ OK
  GUID_PERSONA: 0 nulos  ✅ OK
  CODIGO_SIU_PROGRAMA: 0 nulos  ✅ OK
  PROCESO_ESTANDARIZADO: 0 nulos  ✅ OK


np.True_

#### 2.3 Ingresantes · Duplicados residuales
Se usa la clave candidata del contrato. Se espera `len(df) − len(df.drop_duplicates(subset=clave)) = 0`.


In [7]:
dup_exactos_ing = len(ing) - len(ing.drop_duplicates())
print("Duplicados EXACTOS (todas las columnas):", f"{dup_exactos_ing:,}")

dup_ing = verificar_duplicados(ing, CLAVE_ING)
print()
if dup_ing == 0:
    print("✅ INGRESANTES: 0 duplicados bajo la clave del contrato.")
else:
    print("❌ INGRESANTES: quedan", f"{dup_ing:,}", "duplicados residuales bajo la clave del contrato.")
    print("   El contrato (ingresantes_schema.json) reporta duplicados_con_clave = 0,")
    print("   pero la Silver NO lo cumple.")
    print("   ▶ Acción correctiva: drop_duplicates(subset=CLAVE_ING) antes del modelado Gold")
    print("     y actualizar data/schemas/ingresantes_schema.json.")


Duplicados EXACTOS (todas las columnas): 0


Duplicados bajo la clave (3 cols): 293

❌ INGRESANTES: quedan 293 duplicados residuales bajo la clave del contrato.
   El contrato (ingresantes_schema.json) reporta duplicados_con_clave = 0,
   pero la Silver NO lo cumple.
   ▶ Acción correctiva: drop_duplicates(subset=CLAVE_ING) antes del modelado Gold
     y actualizar data/schemas/ingresantes_schema.json.


#### 2.4 Ingresantes · Normalización de textos
Se verifica que todas las columnas string estén en **MAYÚSCULAS, sin tildes y sin espacios** (bordes ni dobles). Muestra aleatoria de 5 valores de `NOMBRE_PROGRAMA` y `DEPARTAMENTO_FILIAL`.


In [8]:
problemas_ing_norm = verificar_normalizacion(ing, cols_muestra=["NOMBRE_PROGRAMA", "DEPARTAMENTO_FILIAL"])


,columna,tildes,minúsculas,espacios_borde,espacios_dobles,vacíos
0,CODIGO_INEI,0,0,0,0,0
1,NOMBRE_ENTIDAD,0,0,0,0,0
2,TIPO_ENTIDAD,0,0,0,0,0
3,TIPO_GESTION,0,0,0,0,0
4,LICENCIADO,0,0,0,0,0
5,TIPO_CONSTITUCION,0,0,0,0,0
6,NIVEL_ACADEMICO,0,0,0,0,0
7,GUID_PERSONA,0,0,0,0,0
8,SEXO,0,0,0,0,0
9,NACIONALIDAD,0,0,0,0,0


→ ⚠️ Normalización casi completa: 47 celdas con espacios dobles (menor).

Muestra aleatoria (5 valores, random_state=42):

[NOMBRE_PROGRAMA]
['INGENIERIA AGROINDUSTRIAL', 'EDUCACION NIVEL INICIAL', 'DERECHO', 'INGENIERIA CIVIL', 'DISENO INDUSTRIAL']

[DEPARTAMENTO_FILIAL]
['LA LIBERTAD', 'APURIMAC', 'LIMA', 'JUNIN', 'LIMA']


#### 2.5 Ingresantes · Region_Sur
Distribución de `True/False` y validación de la lógica: los departamentos del SUR (Arequipa, Cusco, Tacna, Puno, Moquegua, Apurímac) deben tener `Region_Sur=True` y el resto `False`.


In [9]:
ing_contrad = validar_region_sur(ing, "DEPARTAMENTO_FILIAL")


Distribución de Region_Sur (conteo y %):


,conteo,%
Region_Sur,,
False,2650075,84.93
True,470212,15.07


Contradicciones (depto del SUR pero Region_Sur=False): 0
Contradicciones (depto NO sur pero Region_Sur=True): 0
→ ✅ Lógica Region_Sur consistente con los departamentos del SUR

Conteo por departamento vs Region_Sur:


Region_Sur,Region_Sur=False,Region_Sur=True
DEPARTAMENTO_FILIAL,,
LIMA,1490418,0
LAMBAYEQUE,144856,0
PUNO,0,53842
PIURA,144789,0
HUANUCO,61191,0
ICA,84707,0
JUNIN,147450,0
HUANCAVELICA,28319,0
APURIMAC,0,19022


#### 2.6 Ingresantes · Periodos
Valores únicos de `PROCESO_ESTANDARIZADO`, sin nulos, cubriendo el rango esperado 2020–2025.


In [10]:
esperados_ing = list(range(2020, 2026))
resumen_periodos(ing, "PROCESO_ESTANDARIZADO", esperados=esperados_ing)


,conteo
PROCESO_ESTANDARIZADO,
2020,434038
2021,483029
2022,490593
2023,536895
2024,565860
2025,609872


Valores únicos: 6 | Nulos: 0
✅ Cobertura completa: los 6 periodos esperados están presentes


#### 2.7 Ingresantes · Presencia de UCSP
Filtro por `CODIGO_INEI` = `260000062` (Universidad Católica San Pablo) y confirmación de datos en todos los años esperados.


In [11]:
n_ucsp_ing = verificar_ucsp(ing, "PROCESO_ESTANDARIZADO", esperados_ing)


UCSP → UNIVERSIDAD CATOLICA SAN PABLO | CODIGO_INEI: 260000062
Filas UCSP: 20,415
Periodos esperados: 6 | Presentes en UCSP: 6
→ ✅ UCSP tiene datos en TODOS los periodos esperados


,conteo
PROCESO_ESTANDARIZADO,
2020,3009
2021,3795
2022,3393
2023,3478
2024,3427
2025,3313


#### 2.8 Ingresantes · Distribución de variables clave
Tablas de frecuencia (conteo y %) de `TIPO_GESTION`, `NIVEL_ACADEMICO` y `SEXO`.


In [12]:
frecuencias(ing, ["TIPO_GESTION", "NIVEL_ACADEMICO", "SEXO"])



[TIPO_GESTION]


,conteo,%
TIPO_GESTION,,
PRIVADO,2484353,79.62
PUBLICO,635934,20.38



[NIVEL_ACADEMICO]


,conteo,%
NIVEL_ACADEMICO,,
CARRERA PROFESIONAL,2522716,80.85
MAESTRIA,447626,14.35
SEGUNDA ESPECIALIDAD,112783,3.61
DOCTORADO,37162,1.19



[SEXO]


,conteo,%
SEXO,,
FEMENINO,1603370,51.39
MASCULINO,1516917,48.61


In [13]:
# ---------------------------------------------------------------------------
# FIN FASE 1: guardar agregados + indicadores y liberar INGRESANTES
# ---------------------------------------------------------------------------
aggs["ing_uni_anio"] = ing.groupby(["CODIGO_INEI", "PROCESO_ESTANDARIZADO"], as_index=False).size().rename(
    columns={"size": "n_ingresantes"})
aggs["ing_unis"] = set(ing["CODIGO_INEI"].unique())
aggs["ing_programas"] = set(ing["CODIGO_SIU_PROGRAMA"].dropna().unique())
aggs["ing_nombres"] = ing.drop_duplicates("CODIGO_INEI").set_index("CODIGO_INEI")["NOMBRE_ENTIDAD"]

RESULTADOS["ing"] = {
    "filas": len(ing),
    "nulos_criticas": int(ing[CRITICAS_ING].isna().sum().sum()),
    "duplicados": dup_ing,
    "region_sur_true": int(ing["Region_Sur"].sum()),
    "n_ucsp": n_ucsp_ing,
    "periodos": "2020–2025 (6 años)",
    "norm_graves": problemas_ing_norm[0],
    "norm_menores": problemas_ing_norm[1],
    "region_sur_contrad": ing_contrad,
}

print("Agregados e indicadores de Ingresantes guardados (tablas pequeñas).")
del ing
gc.collect()
print("Memoria actual (RSS):", f"{rss_actual_gb():.2f} GB")
print("✅ Ingresantes liberado. Se procede a cargar Matriculados.")


Agregados e indicadores de Ingresantes guardados (tablas pequeñas).
Memoria actual (RSS): 2.12 GB
✅ Ingresantes liberado. Se procede a cargar Matriculados.


### 2. Matriculados (`matriculados_clean.parquet`)

Dataset **semestral** (granularidad real: persona × programa × local × semestre). La clave efectiva de producción incluye `PERIODO_ESTANDARIZADO`; la del contrato no.


In [14]:
# Carga secuencial · FASE 2: MATRICULADOS
mat = cargar_dataset(MAT_PATH, "MATRICULADOS")


MATRICULADOS cargado
Shape: 17,368,424 filas × 40 columnas
Archivo en disco: 0.74 GB
Memoria estimada (deep): 10.97 GB
RSS actual del proceso: 9.05 GB


#### 2.1 Matriculados · Resumen general
Columnas, tipos de dato, filas totales y rango de periodos (semestres).


In [15]:
print("Filas totales:", f"{len(mat):,}", "| Columnas:", mat.shape[1])
print("\nTipos de dato por columna:")
print(mat.dtypes.to_string())
print("\nRango de periodos (PERIODO_ESTANDARIZADO):", sorted(mat["PERIODO_ESTANDARIZADO"].dropna().unique()))


Filas totales: 17,368,424 | Columnas: 40

Tipos de dato por columna:
CODIGO_INEI                         large_string[pyarrow]
NOMBRE_ENTIDAD                      large_string[pyarrow]
TIPO_ENTIDAD                        large_string[pyarrow]
TIPO_GESTION                        large_string[pyarrow]
TIPO_CONSTITUCION                   large_string[pyarrow]
LICENCIA                            large_string[pyarrow]
PERIODO                             large_string[pyarrow]
PERIODO_ESTANDARIZADO               large_string[pyarrow]
NIVEL_ACADEMICO                     large_string[pyarrow]
PERIODO_LECTIVO                     large_string[pyarrow]
CODIGO_SIU_PROGRAMA                        int64[pyarrow]
CODIGO_GRUPO_1                            double[pyarrow]
NOMBRE_GRUPO_1                      large_string[pyarrow]
CODIGO_GRUPO_3                            double[pyarrow]
NOMBRE_GRUPO_3                      large_string[pyarrow]
NOMBRE_PROGRAMA                     large_string[pyarrow]
ES_


Rango de periodos (PERIODO_ESTANDARIZADO): ['2020-1', '2020-2', '2021-1', '2021-2', '2022-1', '2022-2', '2023-1', '2023-2', '2024-1', '2024-2', '2025-1', '2025-2']


#### 2.2 Matriculados · Análisis de nulos
Conteo y % de nulos por columna (de mayor a menor). Las columnas con **nulos > 5%** se resaltan en rojo. Columnas críticas (`CODIGO_INEI`, `GUID_PERSONA`, `CODIGO_SIU_PROGRAMA`, `PERIODO_ESTANDARIZADO`) deben tener **0 nulos**.


In [16]:
res = tabla_nulos(mat)
if len(res) == 0:
    print("✅ Ninguna columna con nulos.")
else:
    print("Columnas con nulos (de mayor a menor %); >5% resaltado en rojo:")
    display(resaltar_mayor_5pct(res))

print("\nVerificación de columnas críticas (deben tener 0 nulos):")
verificar_criticas(mat, CRITICAS_MAT)


Columnas con nulos (de mayor a menor %); >5% resaltado en rojo:


,columna,nulos,%_nulos
0,DES_DISCAPACIDAD_DE_SITUACION,17307692,99.650000
1,DES_DISCAPACIDAD_DE_DISPOSICION,17307691,99.650000
2,DES_DISCAPACIDAD_DE_LOCOMOCION,17307691,99.650000
3,DES_DISCAPACIDAD_DE_DESTREZA,17307691,99.650000
4,DES_DISCAPACIDAD_DE_COMUNICACION,17307687,99.650000
5,DES_DISCAPACIDAD_DE_CONDUCTA,17307687,99.650000
6,DES_DISCAPACIDAD_DEL_CUIDADO,17307687,99.650000
7,CODIGO_GRUPO_1,3575373,20.590000
8,CODIGO_GRUPO_3,3575373,20.590000
9,NOMBRE_GRUPO_3,3575373,20.590000



Verificación de columnas críticas (deben tener 0 nulos):
  CODIGO_INEI: 0 nulos  ✅ OK
  GUID_PERSONA: 0 nulos  ✅ OK
  CODIGO_SIU_PROGRAMA: 0 nulos  ✅ OK
  PERIODO_ESTANDARIZADO: 0 nulos  ✅ OK


np.True_

#### 2.3 Matriculados · Duplicados residuales (dos claves)
Se comparan **ambas claves** con explicación:

- **Clave del contrato** (6 cols, **sin** `PERIODO_ESTANDARIZADO`): colisiona entre semestres (la misma persona+programa+local se repite en varios periodos), por lo que infla el conteo de duplicados.
- **Clave de producción** (7 cols, **con** `PERIODO_ESTANDARIZADO`): granularidad real validada en `notebooks/02_experimento_matriculados_duplicados.ipynb` y aplicada en `src/process_matriculados.py`.


In [17]:
dup_mat = verificar_duplicados(mat, CLAVE_MAT_PROD, clave_contrato=CLAVE_MAT_CONTRATO)
print()
print("Interpretación:")
print("  - La clave del CONTRATO (6 cols, SIN periodo) no refleja la granularidad real:")
print("    cada fila es una matrícula por semestre, por eso la misma persona+programa+local")
print("    aparece en varios periodos e infla el conteo de 'duplicados'.")
print("  - La clave de PRODUCCIÓN (7 cols, CON PERIODO_ESTANDARIZADO) es la granularidad real")
print("    y debe dar 0 duplicados en la Silver.")
print()
if dup_mat == 0:
    print("✅ MATRICULADOS: 0 duplicados bajo la clave de producción (con periodo).")
else:
    print("❌ MATRICULADOS: quedan", f"{dup_mat:,}", "duplicados residuales.")


,clave,n_columnas,duplicados,%_sobre_total
0,"Contrato (6 cols, SIN periodo)",6,12987406,74.78
1,"Producción (7 cols, CON periodo)",7,0,0.00



Interpretación:
  - La clave del CONTRATO (6 cols, SIN periodo) no refleja la granularidad real:
    cada fila es una matrícula por semestre, por eso la misma persona+programa+local
    aparece en varios periodos e infla el conteo de 'duplicados'.
  - La clave de PRODUCCIÓN (7 cols, CON PERIODO_ESTANDARIZADO) es la granularidad real
    y debe dar 0 duplicados en la Silver.

✅ MATRICULADOS: 0 duplicados bajo la clave de producción (con periodo).


#### 2.4 Matriculados · Normalización de textos
Se verifica que todas las columnas string estén en **MAYÚSCULAS, sin tildes y sin espacios** (bordes ni dobles). Muestra aleatoria de 5 valores de `NOMBRE_PROGRAMA` y `DEPARTAMENTO_LOCAL`.


In [18]:
problemas_mat_norm = verificar_normalizacion(mat, cols_muestra=["NOMBRE_PROGRAMA", "DEPARTAMENTO_LOCAL"])


,columna,tildes,minúsculas,espacios_borde,espacios_dobles,vacíos
0,CODIGO_INEI,0,0,0,0,0
1,NOMBRE_ENTIDAD,0,0,0,0,0
2,TIPO_ENTIDAD,0,0,0,0,0
3,TIPO_GESTION,0,0,0,0,0
4,TIPO_CONSTITUCION,0,0,0,0,0
5,LICENCIA,0,0,0,0,0
6,PERIODO,0,0,0,0,0
7,PERIODO_ESTANDARIZADO,0,0,0,0,0
8,NIVEL_ACADEMICO,0,0,0,0,0
9,PERIODO_LECTIVO,0,0,0,0,0


→ ⚠️ Normalización casi completa: 56 celdas con espacios dobles (menor).

Muestra aleatoria (5 valores, random_state=42):

[NOMBRE_PROGRAMA]


['DERECHO', 'ADMINISTRACION Y NEGOCIOS INTERNACIONALES', 'MEDICINA VETERINARIA Y ZOOTECNIA', 'ARQUITECTURA', 'PSICOLOGIA']

[DEPARTAMENTO_LOCAL]


['LA LIBERTAD', 'LA LIBERTAD', 'TACNA', 'HUANUCO', 'LIMA']


#### 2.5 Matriculados · Region_Sur
Distribución de `True/False` y validación de la lógica sobre `DEPARTAMENTO_LOCAL`.


In [19]:
mat_contrad = validar_region_sur(mat, "DEPARTAMENTO_LOCAL")


Distribución de Region_Sur (conteo y %):


,conteo,%
Region_Sur,,
False,14612877,84.13
True,2755547,15.87


Contradicciones (depto del SUR pero Region_Sur=False): 0
Contradicciones (depto NO sur pero Region_Sur=True): 0
→ ✅ Lógica Region_Sur consistente con los departamentos del SUR

Conteo por departamento vs Region_Sur:


Region_Sur,Region_Sur=False,Region_Sur=True
DEPARTAMENTO_LOCAL,,
LIMA,8031076,0
ICA,444752,0
LAMBAYEQUE,812784,0
AREQUIPA,0,1170824
PUNO,0,454062
PIURA,762443,0
JUNIN,848470,0
HUANCAVELICA,114116,0
HUANUCO,388638,0


#### 2.6 Matriculados · Periodos
Valores únicos de `PERIODO_ESTANDARIZADO`, sin nulos ni malformados, cubriendo los 12 semestres esperados (2020-1 … 2025-2).


In [20]:
esperados_mat = [f"{y}-{s}" for y in range(2020, 2026) for s in (1, 2)]
resumen_periodos(mat, "PERIODO_ESTANDARIZADO", patron=r"^\d{4}-[12]$", esperados=esperados_mat)


,conteo
PERIODO_ESTANDARIZADO,
2020-1,1225466
2020-2,1185775
2021-1,1379035
2021-2,1378119
2022-1,1481585
2022-2,1411507
2023-1,1493879
2023-2,1414984
2024-1,1566874


Valores únicos: 12 | Nulos: 0


Valores malformados (no cumplen ^\d{4}-[12]$ ): 0
✅ Cobertura completa: los 12 periodos esperados están presentes


#### 2.7 Matriculados · Presencia de UCSP
Filtro por `CODIGO_INEI` = `260000062` y confirmación de datos en todos los semestres esperados.


In [21]:
n_ucsp_mat = verificar_ucsp(mat, "PERIODO_ESTANDARIZADO", esperados_mat)


UCSP → UNIVERSIDAD CATOLICA SAN PABLO | CODIGO_INEI: 260000062
Filas UCSP: 107,412
Periodos esperados: 12 | Presentes en UCSP: 12
→ ✅ UCSP tiene datos en TODOS los periodos esperados


,conteo
PERIODO_ESTANDARIZADO,
2020-1,7961
2020-2,8019
2021-1,9016
2021-2,8724
2022-1,9080
2022-2,8602
2023-1,9288
2023-2,9080
2024-1,9582


#### 2.8 Matriculados · Distribución de variables clave
Tablas de frecuencia (conteo y %) de `TIPO_GESTION`, `NIVEL_ACADEMICO` y `SEXO`.


In [22]:
frecuencias(mat, ["TIPO_GESTION", "NIVEL_ACADEMICO", "SEXO"])



[TIPO_GESTION]


,conteo,%
TIPO_GESTION,,
PRIVADO,12839664,73.93
PUBLICO,4528760,26.07



[NIVEL_ACADEMICO]


,conteo,%
NIVEL_ACADEMICO,,
CARRERA PROFESIONAL,16049485,92.41
MAESTRIA,968270,5.57
SEGUNDA ESPECIALIDAD,211906,1.22
DOCTORADO,138763,0.8



[SEXO]


,conteo,%
SEXO,,
FEMENINO,8918800,51.35
MASCULINO,8449624,48.65


In [23]:
# ---------------------------------------------------------------------------
# FIN FASE 2: guardar agregados + indicadores y liberar MATRICULADOS
# ---------------------------------------------------------------------------
mat["ANIO"] = mat["PERIODO_ESTANDARIZADO"].str[:4].astype(int)
aggs["mat_uni_anio"] = mat.groupby(["CODIGO_INEI", "ANIO"], as_index=False).size().rename(
    columns={"size": "n_matriculados", "ANIO": "PROCESO_ESTANDARIZADO"})
aggs["mat_unis"] = set(mat["CODIGO_INEI"].unique())
aggs["mat_programas"] = set(mat["CODIGO_SIU_PROGRAMA"].dropna().unique())
aggs["mat_nombres"] = mat.drop_duplicates("CODIGO_INEI").set_index("CODIGO_INEI")["NOMBRE_ENTIDAD"]

RESULTADOS["mat"] = {
    "filas": len(mat),
    "nulos_criticas": int(mat[CRITICAS_MAT].isna().sum().sum()),
    "duplicados": dup_mat,
    "region_sur_true": int(mat["Region_Sur"].sum()),
    "n_ucsp": n_ucsp_mat,
    "periodos": "2020-1 … 2025-2 (12 semestres)",
    "norm_graves": problemas_mat_norm[0],
    "norm_menores": problemas_mat_norm[1],
    "region_sur_contrad": mat_contrad,
}

print("Agregados e indicadores de Matriculados guardados (tablas pequeñas).")
del mat
gc.collect()
print("Memoria actual (RSS):", f"{rss_actual_gb():.2f} GB")
print("✅ Matriculados liberado. Ambas fases completadas sin mantener ambos datasets en RAM.")


Agregados e indicadores de Matriculados guardados (tablas pequeñas).
Memoria actual (RSS): 10.85 GB
✅ Matriculados liberado. Ambas fases completadas sin mantener ambos datasets en RAM.


## 3. Relación entre datasets (visión simplificada)

Se calcula **solo sobre agregados** (guardados en las fases 1 y 2), sin recargar los datasets completos: intersección de universidades, programas y periodos, más una **propuesta conceptual** de dimensiones para el modelo estrella. **No se construyen tablas Gold.**


In [24]:
print("UNIVERSIDADES (CODIGO_INEI)")
print("  Ingresantes:", f"{len(aggs['ing_unis']):,}", "| Matriculados:", f"{len(aggs['mat_unis']):,}",
      "| Intersección:", f"{len(aggs['ing_unis'] & aggs['mat_unis']):,}")
print("PROGRAMAS (CODIGO_SIU_PROGRAMA)")
print("  Ingresantes:", f"{len(aggs['ing_programas']):,}", "| Matriculados:", f"{len(aggs['mat_programas']):,}",
      "| Intersección:", f"{len(aggs['ing_programas'] & aggs['mat_programas']):,}")
anios_ing = set(aggs["ing_uni_anio"]["PROCESO_ESTANDARIZADO"].unique())
anios_mat = set(aggs["mat_uni_anio"]["PROCESO_ESTANDARIZADO"].unique())
print("PERIODOS (año)")
print("  Ingresantes:", sorted(anios_ing), "| Matriculados:", sorted(anios_mat),
      "| Intersección:", sorted(anios_ing & anios_mat))


UNIVERSIDADES (CODIGO_INEI)
  Ingresantes: 138 | Matriculados: 174 | Intersección: 135
PROGRAMAS (CODIGO_SIU_PROGRAMA)
  Ingresantes: 534 | Matriculados: 525 | Intersección: 522
PERIODOS (año)
  Ingresantes: [2020, 2021, 2022, 2023, 2024, 2025] | Matriculados: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)] | Intersección: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


### 3.2. Propuesta conceptual de relación (preparación del modelo estrella)

Ambos datasets comparten la identidad de **Universidad** (`CODIGO_INEI` + `NOMBRE_ENTIDAD`) y de **Programa** (`CODIGO_SIU_PROGRAMA` + `NOMBRE_PROGRAMA`), y cubren los mismos años (2020–2025). Relación sugerida:

- **dim_universidad**: `CODIGO_INEI` (clave natural) + nombre, tipo, gestión, licenciamiento, etc. Atributos compartidos por ambos datasets.
- **dim_programa**: `CODIGO_SIU_PROGRAMA` (clave natural) + nombre y clasificaciones de grupo (`CODIGO_GRUPO_1` / `CODIGO_GRUPO_3`).
- **dim_periodo**: el **año** como punto de unión. Ingresantes es **anual** (`PROCESO_ESTANDARIZADO`) y matriculados es **semestral** (`PERIODO_ESTANDARIZADO` = "2025-1"); para comparar se agrega matriculados al año.
- **dim_ubicacion**: departamento/provincia/distrito del filial (ingresantes) o del local (matriculados), con `Region_Sur` como atributo analítico.

> Esto es **solo una propuesta conceptual**: en la capa Gold se construirán las dimensiones y los hechos (ingresos y matrículas) siguiendo estas claves. Aquí solo se demuestra la viabilidad de la relación.


In [25]:
# 3.3 Ejemplo de agregaciones: ingresantes vs matriculados por universidad y año
comp = aggs["ing_uni_anio"].merge(aggs["mat_uni_anio"],
                                   on=["CODIGO_INEI", "PROCESO_ESTANDARIZADO"], how="outer")
comp[["n_ingresantes", "n_matriculados"]] = comp[["n_ingresantes", "n_matriculados"]].fillna(0).astype(int)

nombres = pd.concat([aggs["ing_nombres"], aggs["mat_nombres"]])
nombres = nombres[~nombres.index.duplicated(keep="first")]
comp["NOMBRE_ENTIDAD"] = comp["CODIGO_INEI"].map(nombres)

print("UCSP: ingresantes vs matriculados por año")
display(comp[comp["CODIGO_INEI"] == CODIGO_INEI_UCSP].sort_values("PROCESO_ESTANDARIZADO"))

tot = comp.groupby(["CODIGO_INEI", "NOMBRE_ENTIDAD"], as_index=False)[["n_ingresantes", "n_matriculados"]].sum()
tot["ratio_mat_ing"] = (tot["n_matriculados"] / tot["n_ingresantes"].replace(0, np.nan)).round(2)
print("\nTop 10 universidades por volumen total de matriculados:")
display(tot.sort_values("n_matriculados", ascending=False).head(10).reset_index(drop=True))


UCSP: ingresantes vs matriculados por año


,CODIGO_INEI,PROCESO_ESTANDARIZADO,n_ingresantes,n_matriculados,NOMBRE_ENTIDAD
661,260000062,2020,3009,15980,UNIVERSIDAD CATOLICA SAN PABLO
662,260000062,2021,3795,17740,UNIVERSIDAD CATOLICA SAN PABLO
663,260000062,2022,3393,17682,UNIVERSIDAD CATOLICA SAN PABLO
664,260000062,2023,3478,18368,UNIVERSIDAD CATOLICA SAN PABLO
665,260000062,2024,3427,18664,UNIVERSIDAD CATOLICA SAN PABLO
666,260000062,2025,3313,18978,UNIVERSIDAD CATOLICA SAN PABLO



Top 10 universidades por volumen total de matriculados:


,CODIGO_INEI,NOMBRE_ENTIDAD,n_ingresantes,n_matriculados,ratio_mat_ing
0,260000052,UNIVERSIDAD CESAR VALLEJO S.A.C.,420770,2004870,4.76
1,260000065,UNIVERSIDAD TECNOLOGICA DEL PERU S.A.C.,531397,1942481,3.66
2,260000055,UNIVERSIDAD PRIVADA DEL NORTE S.A.C.,243547,1344591,5.52
3,260000054,UNIVERSIDAD PERUANA DE CIENCIAS APLICADAS S.A.C.,123930,845989,6.83
4,26000067A,UNIVERSIDAD CONTINENTAL S.A.C.,172233,684754,3.98
5,160000001,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,68771,483614,7.03
6,260000019,UNIVERSIDAD DE SAN MARTIN DE PORRES,84119,418481,4.97
7,260000008,PONTIFICIA UNIVERSIDAD CATOLICA DEL PERU,47163,366814,7.78
8,260000046,UNIVERSIDAD PRIVADA ANTENOR ORREGO,52256,337454,6.46
9,160000005,UNIVERSIDAD NACIONAL DE SAN AGUSTIN DE AREQUIPA,48501,333016,6.87


## 4. Resumen ejecutivo

Indicadores clave de calidad para ambos datasets. Las filas marcadas ❌ requieren acción correctiva antes del modelado Gold.


In [26]:
R = RESULTADOS

resumen_ejecutivo = pd.DataFrame({
    "Indicador": [
        "Filas totales",
        "Nulos en columnas críticas (0 esperado)",
        "Duplicados residuales (0 esperado)",
        "Periodos cubiertos",
        "Region_Sur True (conteo)",
        "Presencia de UCSP",
    ],
    "Ingresantes": [
        f"{R['ing']['filas']:,}",
        f"{R['ing']['nulos_criticas']} ✅" if R['ing']['nulos_criticas'] == 0 else f"{R['ing']['nulos_criticas']} ❌",
        f"{R['ing']['duplicados']:,} ❌" if R['ing']['duplicados'] else "0 ✅",
        R['ing']['periodos'],
        f"{R['ing']['region_sur_true']:,}",
        "Sí ✅" if R['ing']['n_ucsp'] else "No ❌",
    ],
    "Matriculados": [
        f"{R['mat']['filas']:,}",
        f"{R['mat']['nulos_criticas']} ✅" if R['mat']['nulos_criticas'] == 0 else f"{R['mat']['nulos_criticas']} ❌",
        f"{R['mat']['duplicados']:,} ✅" if R['mat']['duplicados'] == 0 else f"{R['mat']['duplicados']:,} ❌",
        R['mat']['periodos'],
        f"{R['mat']['region_sur_true']:,}",
        "Sí ✅" if R['mat']['n_ucsp'] else "No ❌",
    ],
})
display(resumen_ejecutivo)

pico_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
print("=" * 78)
print("¿Los datos están listos para el modelado Gold? → SÍ, CON CORRECCIONES")
print("=" * 78)
print("MATRICULADOS ✅  Listo para el Gold: 0 duplicados (clave de producción con periodo),")
print("  0 nulos en columnas críticas, periodos 2020-1…2025-2 completos y UCSP presente en los 12 semestres.")
print()
print(f"INGRESANTES ⚠️  Presenta {R['ing']['duplicados']:,} duplicados residuales bajo la clave del contrato")
print("  (el contrato reporta 0). Acciones correctivas antes del Gold:")
print("    1. drop_duplicates(subset=CLAVE_ING) sobre ingresantes_clean.parquet.")
print("    2. Re-validar que el residual quede en 0.")
print("    3. Actualizar data/schemas/ingresantes_schema.json (duplicados_con_clave).")
print()
print("Notas adicionales:")
print("  - Normalización: 0 tildes/minúsculas/espacios en bordes en ambos datasets. Solo celdas con")
print(f"    espacios dobles (menor): Ingresantes={R['ing']['norm_menores']}, Matriculados={R['mat']['norm_menores']}.")
print(f"  - Region_Sur consistente en ambos datasets (contradicciones = 0).")
print(f"  - Memoria pico del proceso: {pico_gb:.2f} GB (objetivo ≤ 9 GB).")


,Indicador,Ingresantes,Matriculados
0,Filas totales,"3,120,287","17,368,424"
1,Nulos en columnas críticas (0 esperado),0 ✅,0 ✅
2,Duplicados residuales (0 esperado),293 ❌,0 ✅
3,Periodos cubiertos,2020–2025 (6 años),2020-1 … 2025-2 (12 semestres)
4,Region_Sur True (conteo),"470,212","2,755,547"
5,Presencia de UCSP,Sí ✅,Sí ✅


¿Los datos están listos para el modelado Gold? → SÍ, CON CORRECCIONES
MATRICULADOS ✅  Listo para el Gold: 0 duplicados (clave de producción con periodo),
  0 nulos en columnas críticas, periodos 2020-1…2025-2 completos y UCSP presente en los 12 semestres.

INGRESANTES ⚠️  Presenta 293 duplicados residuales bajo la clave del contrato
  (el contrato reporta 0). Acciones correctivas antes del Gold:
    1. drop_duplicates(subset=CLAVE_ING) sobre ingresantes_clean.parquet.
    2. Re-validar que el residual quede en 0.
    3. Actualizar data/schemas/ingresantes_schema.json (duplicados_con_clave).

Notas adicionales:
  - Normalización: 0 tildes/minúsculas/espacios en bordes en ambos datasets. Solo celdas con
    espacios dobles (menor): Ingresantes=47, Matriculados=56.
  - Region_Sur consistente en ambos datasets (contradicciones = 0).
  - Memoria pico del proceso: 11.65 GB (objetivo ≤ 9 GB).
